05 - What about the baseline model? 
For a time-based prediction problem, a simple baseline model that predicts the training mean is not the best choice. 
There are other possibilities:
-	Predict historical training mean (weak reference)
-	Persistence model: usage (t) = usage (t-1). Assume the next interval looks like the previous one. If your ML model cannot beat this, you have a problem.
-	Same time yesterday
-	Recent rolling mean (e.g. average usage during previous hour)

Your ML model must beat all these baseline models.
What about the metrics?
MAE, RMSE, R2. The MAE is the most straightforward one. 
Every model must see the same folds, for a rigorous comparison.
We will use expanding window, instead of rolling window:
-	Expanding window: amount of training data grows. 
Jan-Apr → May
Jan-May → Jun
Jan-Jun → Jul
...
-	Rolling window: Old data gets dropped
Jan-Apr → May
Feb-May → Jun
Mar-Jun → Jul
Rolling window becomes more relevant if the system changes over time. You can avoid that old settings/setups still have an influence. 

Notebook operations:
1.	Load your golden table
2.	Make sure it’s time ordered
3.	Make sure the baseline features exist
4.	Define the 4 baselines
Historical mean in the training period
Prediction based on the previous 15 min
Prediction of same time yesterday
Prediction of previous hour average
5.	Create one metric function to calculate (MAE, RMSE, R2)
6.	Define the backtesting folds (
folds = [ ("2018-05-01", "2018-06-01"), ("2018-06-01", "2018-07-01"), ("2018-07-01", "2018-08-01"), ("2018-08-01", "2018-09-01"), ("2018-09-01", "2018-10-01"), ("2018-10-01", "2018-11-01"), ("2018-11-01", "2018-12-01"), ]
Each pair is validation start and validation end date. In this way, all the previous dates are training dates.
7.	Walk forward: 
For each fold, you define the training and the validation sets. And you evaluate the baseline models with the metrics defined. 
8.	Summarize performance across folds (you will have to average the results of each fold)
9.	Check errors. E.g. plot MAE by month. If the prediction gets worse all of a sudden there might have been a process change.


In [14]:
#Connecting to database
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

con = duckdb.connect("Steel_energy.duckdb")

In [2]:
#Loading table to pandas

In [3]:
#let's load it into pandas
df = con.sql("""
SELECT *
FROM gold_energy_features
ORDER BY timestamp
""").df()

In [4]:
#Make sure it's ordered

In [5]:
df = df.sort_values("timestamp").reset_index(drop=True)

In [6]:
df["timestamp"].min(), df["timestamp"].max()

(Timestamp('2018-01-01 00:00:00'), Timestamp('2018-12-31 23:45:00'))

In [7]:
#Make sure the baseline features exist

In [8]:
df.columns

Index(['timestamp', 'usage_kwh', 'hour', 'month', 'day_of_week', 'week_status',
       'load_type', 'usage_15min_ago', 'usage_1h_ago', 'usage_lag_96',
       'avg_usage_previous_hour'],
      dtype='str')

In [9]:
#Create one metric function
def evaluate_forecast(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    
    r2 = r2_score(y_true, y_pred)
    
    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [10]:
#Define the backtesting folds
folds = [
    ("2018-05-01", "2018-06-01"),
    ("2018-06-01", "2018-07-01"),
    ("2018-07-01", "2018-08-01"),
    ("2018-08-01", "2018-09-01"),
    ("2018-09-01", "2018-10-01"),
    ("2018-10-01", "2018-11-01"),
    ("2018-11-01", "2018-12-01"),
]
#The training data will simply be everything before validation_start

In [16]:
results = []

for fold_number, (val_start, val_end) in enumerate(folds, start=1):

    train = df[
        df["timestamp"] < val_start
    ].copy()

    val = df[
        (df["timestamp"] >= val_start) &
        (df["timestamp"] < val_end)
    ].copy()

    # -------------------------
    # Baseline 0: Training mean
    # -------------------------

    train_mean = train["usage_kwh"].mean()

    mean_pred = np.full(
        len(val),
        train_mean
    )

    metrics = evaluate_forecast(
        val["usage_kwh"],
        mean_pred
    )

    results.append({
        "fold": fold_number,
        "validation_month": val_start,
        "model": "Training Mean",
        **metrics
    })


    # -------------------------
    # Baseline 1: Persistence
    # Predict current usage
    # using usage 15 minutes ago
    # -------------------------

    temp = val.dropna(
        subset=["usage_15min_ago"]
    )

    metrics = evaluate_forecast(
        temp["usage_kwh"],
        temp["usage_15min_ago"]
    )

    results.append({
        "fold": fold_number,
        "validation_month": val_start,
        "model": "Persistence t-1",
        **metrics
    })


    # -------------------------
    # Baseline 2: Same time yesterday
    # -------------------------

    temp = val.dropna(
        subset=["usage_lag_96"]
    )

    metrics = evaluate_forecast(
        temp["usage_kwh"],
        temp["usage_lag_96"]
    )

    results.append({
        "fold": fold_number,
        "validation_month": val_start,
        "model": "Same Time Yesterday t-96",
        **metrics
    })


    # -------------------------
    # Baseline 3: Previous-hour mean
    # -------------------------

    temp = val.dropna(
        subset=["avg_usage_previous_hour"]
    )

    metrics = evaluate_forecast(
        temp["usage_kwh"],
        temp["avg_usage_previous_hour"]
    )

    results.append({
        "fold": fold_number,
        "validation_month": val_start,
        "model": "Previous Hour Mean",
        **metrics
    })

In [17]:
#Check the results

In [18]:
results_df = pd.DataFrame(results)

results_df

,fold,validation_month,model,MAE,RMSE,R2
0,1,2018-05-01,Training Mean,30.721367,34.481009,-0.032716
1,1,2018-05-01,Persistence t-1,6.322833,14.310939,0.822108
2,1,2018-05-01,Same Time Yesterday t-96,16.602184,30.039460,0.216200
3,1,2018-05-01,Previous Hour Mean,8.956285,18.574421,0.700324
4,2,2018-06-01,Training Mean,27.475853,30.612782,-0.088587
5,2,2018-06-01,Persistence t-1,5.579962,12.700826,0.812621
6,2,2018-06-01,Same Time Yesterday t-96,14.760312,28.030404,0.087324
7,2,2018-06-01,Previous Hour Mean,8.179787,16.763563,0.673570
8,3,2018-07-01,Training Mean,28.108155,31.386359,-0.006650
9,3,2018-07-01,Persistence t-1,6.132339,12.806106,0.832417


In [19]:
#Summarize with averaging by baseline model

In [20]:
baseline_summary = (
    results_df
    .groupby("model")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std")
    )
    .sort_values("MAE_mean")
)

baseline_summary

,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
model,,,,,,
Persistence t-1,5.986253,0.846897,13.249100,1.203816,0.824766,0.014016
Previous Hour Mean,8.727136,1.203899,17.759898,1.567207,0.685384,0.018717
Same Time Yesterday t-96,15.265283,1.956272,28.236313,2.814162,0.200698,0.109136
Training Mean,28.526176,1.451200,32.220573,1.880668,-0.038730,0.039684


Walk-forward evaluation showed that the strongest naive forecast was the 15-minute persistence baseline, achieving a mean MAE of approximately 5.99 kWh and mean R² of 0.825 across seven validation months. The substantially weaker 24-hour lag indicates that short-term process continuity is considerably more informative than simple daily periodicity. The next modeling stage will therefore test whether machine-learning models can improve on short-term persistence by combining recent energy history, calendar information, and other prediction-time-available process variables.